In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

In [ ]:
load_dotenv()

ANNOTATIONS_ENDPOINT = "https://ooinet.oceanobservatories.org/api/m2m/12580/anno/find"

OOI_USERNAME = os.environ["OOI_USERNAME"]
OOI_TOKEN = os.environ["OOI_TOKEN"]

In [ ]:
PROFILER_SITES = {
    "oregon_offshore":      "CE04OSPS",
    "oregon_offshore_deep": "CE04OSPD",
    "slope_base":           "RS01SBPS",
    "slope_base_deep":      "RS01SBPD",
    "axial_base":           "RS03AXPS",
    "axial_base_deep":      "RS03AXPD",
}

In [ ]:
def harvest_annotations(subsite: str) -> pd.DataFrame:
    """Fetch all HITL annotations for a subsite from the OOI M2M API.

    Parameters
    ----------
    subsite:
        OOI subsite code (e.g. "CE04OSPS") or a site key from PROFILER_SITES
        (e.g. "oregon_offshore").

    Returns
    -------
    pd.DataFrame with one row per annotation, sorted by beginDT.
    """
    subsite = PROFILER_SITES.get(subsite, subsite)

    response = requests.get(
        ANNOTATIONS_ENDPOINT,
        params={"refdes": subsite},
        auth=(OOI_USERNAME, OOI_TOKEN),
    )
    response.raise_for_status()

    records = response.json()
    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)

    for col in ("beginDT", "endDT"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], unit="ms", utc=True)

    return df.sort_values("beginDT").reset_index(drop=True)

In [ ]:
subsite = "oregon_offshore"

df = harvest_annotations(subsite)
print(f"{len(df)} annotations for {PROFILER_SITES.get(subsite, subsite)}")
df

In [ ]:
out_path = Path("../annotations") / f"{PROFILER_SITES.get(subsite, subsite)}.csv"
out_path.parent.mkdir(exist_ok=True)
df.to_csv(out_path, index=False)
print(f"saved {len(df)} annotations to {out_path}")